# ✦ A/B Testing para Evaluar Estrategias de Prompting

**Materiales desarrollados para el IFTS24 — 2026**

**Espacio:** Laboratorio de PLN (Procesamiento del Lenguaje Natural)

---

## Objetivo

Comparar científicamente la efectividad de diferentes estrategias de prompting (zero-shot vs. few-shot) mediante un experimento de A/B testing con evaluación humana ciega usando la API de Gemini.

## Al terminar este material vas a poder:

1. Explicar la diferencia práctica entre zero-shot y few-shot prompting.
2. Diseñar y ejecutar un experimento de A/B testing para evaluar respuestas de modelos de lenguaje.
3. Construir una interfaz interactiva de evaluación humana ciega usando `ipywidgets`.
4. Analizar los resultados de tu experimento usando estadísticas descriptivas básicas con `pandas` y visualizarlos gráficamente.


## 1. Introducción y Conceptos Teóricos

> **El caso de hoy**
> ¿Alguna vez te preguntaste si realmente vale la pena el esfuerzo de redactar ejemplos (few-shot) para guiar a un modelo de lenguaje, o si un prompt directo y simple (zero-shot) obtiene el mismo resultado? Hoy vas a diseñar y ejecutar un experimento científico de A/B testing con evaluación a ciegas para responder a esta pregunta con datos reales.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **A/B Testing** | Un método experimental para comparar dos versiones (A y B) de una variable y determinar cuál rinde mejor. |
| **Zero-shot prompting** | Pedirle al modelo que resuelva una tarea de forma directa, basándose únicamente en la instrucción y sin darle ningún ejemplo previo. |
| **Few-shot prompting** | Proporcionarle al modelo algunos ejemplos resueltos de la tarea antes de pedirle que resuelva el caso de interés. |
| **Evaluación ciega** | Calificar las respuestas sin conocer qué variante de prompt las generó, para evitar que el evaluador tenga sesgos o preferencias. |
| **p-valor (p-value)** | Medida estadística de probabilidad que nos indica si la diferencia entre dos resultados es real (estadísticamente significativa) o si se debe simplemente al azar. |


## 2. Configuración del Experimento

### Instalación y Configuración de la API

Primero, configuramos el entorno para utilizar la API de Google Gemini.

In [ ]:
# ── Instalación de Librerías (Condicional para Colab) ──
# Si ejecutás en Google Colab, se instalarán el SDK oficial de Google GenAI y las librerías necesarias.
import sys
if 'google.colab' in sys.modules:
    print("☁️ Ejecutando en Google Colab. Instalando paquetes...")
    !pip install -q google-genai pandas matplotlib ipywidgets scipy python-dotenv
    print("✓ Librerías instaladas.")
else:
    print("💻 Ejecutando en entorno local. Asegurate de tener instalados los paquetes necesarios.")

In [ ]:
# ── Inicialización y Configuración de la API ──
import os
import sys
import pandas as pd

# Detectar el entorno
if 'google.colab' in sys.modules:
    from google.colab import userdata
    # Obtenemos la API Key desde los secretos de Colab
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY') or userdata.get('GEMINI_API_KEY')
else:
    # Intentamos cargar variables desde el archivo .env local
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    # Leemos la API Key de las variables de entorno
    GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')

from google import genai
from google.genai import types

# Inicializar el cliente oficial de Google GenAI
if GOOGLE_API_KEY:
    cliente = genai.Client(api_key=GOOGLE_API_KEY)
    print("✓ Cliente de Gemini inicializado correctamente usando API Key.")
else:
    try:
        cliente = genai.Client()
        print("✓ Cliente de Gemini inicializado con credenciales del entorno.")
    except Exception as e:
        print("❌ Error: No se encontró la API Key de Gemini en los secretos de Colab ni en el archivo .env.")
        print(f"Detalle: {e}")

# Definimos el modelo a utilizar
MODEL_ID = "gemini-2.5-flash"
print(f"✓ Modelo configurado: {MODEL_ID}")

### Parámetros del Experimento

- **Número de iteraciones**: 5 por cada variante
- **Tarea**: Generar nombres creativos para productos
- **Métrica de evaluación**: Calificación humana (thumbs up/down)

## 3. Diseño de Prompts

### Variante A: Zero-shot

En esta variante, proporcionamos solo la descripción del producto y las palabras clave, sin ejemplos previos.

In [ ]:
# Variante A: Zero-shot prompting
prompt_A = """Descripción del producto: Un par de zapatos que pueden adaptarse a cualquier talla de pie.
Palabras clave: adaptable, ajuste, omni-fit.
Nombres de productos:"""

print("📝 Prompt A (Zero-shot):")
print("-" * 50)
print(prompt_A)

### Variante B: Few-shot

En esta variante, proporcionamos dos ejemplos completos antes de solicitar la tarea principal.

In [ ]:
# Variante B: Few-shot prompting
prompt_B = """Descripción del producto: Una licuadora casera para batidos.
Palabras clave: rápido, saludable, compacto.
Nombres de productos: BatidoCasa, FitBatidor, RápidoBatido, BatidoraCasera

Descripción del producto: Un reloj que puede dar la hora exacta en el espacio.
Palabras clave: astronauta, resistente al espacio, órbita elíptica
Nombres de productos: AstroTiempo, GuardiaEspacial, PrecisiónOrbital, TiempoElíptico.

Descripción del producto: Un par de zapatos que pueden adaptarse a cualquier talla de pie.
Palabras clave: adaptable, ajuste, omni-fit.
Nombres de productos:"""

print("📝 Prompt B (Few-shot):")
print("-" * 50)
print(prompt_B)

### Diferencias Clave

1. **Prompt A** espera que el modelo entienda la tarea sin ejemplos
2. **Prompt B** proporciona un patrón claro con dos ejemplos completos
3. Ambos solicitan la misma tarea final: nombres para zapatos adaptables

## 4. Ejecución del Experimento

### Función de Generación de Respuestas

Definimos una función que interactúa con el modelo de Gemini.

In [ ]:
def get_response(prompt):
    """
    Se conecta con la API de Gemini para generar una respuesta basada en el prompt.
    """
    # Enviamos la solicitud de generación al modelo configurado
    respuesta = cliente.models.generate_content(
        model=MODEL_ID,
        contents=[prompt]
    )
    # Devolvemos únicamente el texto de la respuesta generada
    return respuesta.text

print("✓ Función get_response lista para usarse.")

### Generación y Almacenamiento de Respuestas

Ejecutamos el experimento generando múltiples respuestas para cada variante.

In [ ]:
# Definimos la lista de prompts que vamos a probar
lista_prompts = [prompt_A, prompt_B]

# Configuración de las repeticiones del experimento
respuestas_guardadas = []
cantidad_pruebas = 5  # Hacemos 5 iteraciones por variante para tener una muestra estadística

print("🚀 Iniciando experimento A/B Testing...\n")

# Recorremos cada prompt para generar sus respuestas correspondientes
for posicion, prompt in enumerate(lista_prompts):
    # Asignamos la letra de la variante según la posición
    if posicion == 0:
        variante_letra = "A"
    else:
        variante_letra = "B"
        
    print(f"📊 Generando respuestas para Variante {variante_letra}...")

    for repeticion in range(cantidad_pruebas):
        # Solicitamos la respuesta del modelo llamando a nuestra función
        texto_respuesta = get_response(prompt)

        # Estructuramos los datos recolectados en un diccionario para la tabla
        registro_datos = {
            "variante": variante_letra,
            "prompt": prompt,
            "respuesta": texto_respuesta
        }
        respuestas_guardadas.append(registro_datos)
        
        # Mostramos el progreso por pantalla
        nro_respuesta = repeticion + 1
        print(f"  ✓ Respuesta {nro_respuesta}/{cantidad_pruebas} generada.")

    print() # Línea en blanco entre variantes
    
print("✓ Todas las respuestas fueron generadas con éxito.")

### Visualización Inicial de Datos

Convertimos las respuestas en un DataFrame para facilitar el análisis.

In [ ]:
# Convertimos la lista de diccionarios en un DataFrame de Pandas
df = pd.DataFrame(respuestas_guardadas)

# Guardamos el conjunto de respuestas en un archivo CSV para persistir los datos
df.to_csv("respuestas.csv", index=False)

total_generado = len(df)
print("📄 Resumen del experimento:")
print(f"✓ Total de respuestas generadas en la tabla: {total_generado}")
print(f"✓ Respuestas por variante: {cantidad_pruebas}")
print("\n📊 Vista previa de las primeras filas generadas:")
print(df.head())

In [ ]:
# Mostrar la distribución de respuestas para validar que esté balanceado
print("\n📈 Distribución de respuestas por variante:")
conteo_variantes = df['variante'].value_counts()
print(conteo_variantes)

# Extraemos un ejemplo de cada variante para inspeccionar su estilo
print("\n💡 Ejemplos de respuestas generadas:")

# Primero filtramos por la variante A y extraemos la primera respuesta
tabla_variante_A = df[df['variante'] == 'A']
primera_respuesta_A = tabla_variante_A['respuesta'].iloc[0]
print("\nVariante A (Zero-shot):")
print(primera_respuesta_A)

# Hacemos lo mismo para la variante B
tabla_variante_B = df[df['variante'] == 'B']
primera_respuesta_B = tabla_variante_B['respuesta'].iloc[0]
print("\nVariante B (Few-shot):")
print(primera_respuesta_B)

### 🐛 Laboratorio de Rotura

A veces queremos saber rápidamente cuántos registros corresponden a una condición en un DataFrame. En Pandas, es común ver que las personas intentan filtrar la tabla y medir su tamaño usando la función nativa `len()`.

El código de abajo *parece* correcto. Antes de correrlo, predecí: si queremos contar cuántas filas corresponden a la **Variante A** (que sabemos que son 5), ¿qué va a imprimir la celda?

- ¿Va a imprimir `5`?
- ¿Or va a imprimir `10`?

Formulá tu hipótesis antes de correr la celda.

In [ ]:
# ── Momento 1 — El error común al contar filas ──
# Creamos la condición booleana para la Variante A
condicion_A = df["variante"] == "A"

# Intentamos contar cuántos registros cumplen esta condición usando len()
print("¿Qué devuelve len(df['variante'] == 'A')?")
resultado_erroneo = len(condicion_A)
print(f"Resultado: {resultado_erroneo}")

# Pensá antes de mirar abajo: ¿por qué devolvió 10 en lugar de 5?

### ◈ De la serie completa al filtro real

El error ocurre porque la expresión `df["variante"] == "A"` devuelve una Serie de Pandas con valores booleanos (`True`/`False`) para **cada una** de las filas de la tabla. Por ende, sigue teniendo 10 elementos. Al aplicarle `len()`, medís el tamaño total de la Serie, no la cantidad de valores `True`.

Para solucionarlo, tenés dos caminos correctos:
1. **Filtrar la tabla primero** y luego medir el largo de la tabla filtrada.
2. **Sumar la serie booleana**, ya que en Python `True` cuenta como `1` y `False` como `0`.

In [ ]:
# ── Momento 2 — La solución correcta ──

# Método 1: Filtrar el DataFrame completo y luego medir su longitud
tabla_filtrada_A = df[df["variante"] == "A"]
cantidad_metodo1 = len(tabla_filtrada_A)
print(f"✓ Método 1 (Filtrar y medir): {cantidad_metodo1} filas")

# Método 2: Sumar los valores True de la serie booleana
cantidad_metodo2 = (df["variante"] == "A").sum()
print(f"✓ Método 2 (Sumar Trues): {cantidad_metodo2} filas")

## 5. Evaluación Interactiva

### Interfaz de Evaluación Humana

Creamos una interfaz interactiva para que los evaluadores califiquen las respuestas de manera ciega (sin saber qué variante están evaluando).

In [ ]:
!uv pip install ipywidgets

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

# Cargamos el archivo de respuestas guardadas previamente
tabla_respuestas = pd.read_csv("respuestas.csv")

# Mezclamos la tabla de manera aleatoria para asegurar una evaluación a ciegas (blind test)
# Hacemos esto en dos pasos para evitar encadenar operaciones
tabla_desordenada = tabla_respuestas.sample(frac=1.0)
df = tabla_desordenada.reset_index(drop=True)

print("🔀 Respuestas mezcladas aleatoriamente para evaluación ciega.")

# Inicializamos la variable de control que rastreará en qué número de respuesta estamos
response_index = 0

# Creamos una columna vacía para registrar las calificaciones humanas (thumbs up/down)
df['feedback'] = pd.Series(dtype='float') 
print("✓ Columna 'feedback' inicializada en la tabla.")

### Proceso de Calificación

Los evaluadores verán cada respuesta y podrán calificarla con:
- 👍 (Pulgar arriba): Respuesta de buena calidad
- 👎 (Pulgar abajo): Respuesta de baja calidad

In [ ]:
def on_button_clicked(b):
    """
    Maneja el evento de clic en los botones de evaluación de la interfaz interactiva.
    """
    global response_index
    
    # Si hace clic en 👍 registramos un 1 (Positivo), si es 👎 registramos un 0 (Negativo)
    if b.description == "👍":
        calificacion = 1
    else:
        calificacion = 0

    # Guardamos la calificación en la fila correspondiente
    df.at[response_index, 'feedback'] = calificacion

    # Pasamos a la siguiente respuesta
    response_index += 1
    
    # Si quedan respuestas pendientes por calificar, actualizamos la pantalla
    if response_index < len(df):
        update_response()
    else:
        # Al finalizar, guardamos la tabla con calificaciones en un archivo nuevo
        df.to_csv("resultados.csv", index=False)
        print("\n✓ ¡Calificación finalizada! Resultados guardados en 'resultados.csv'.")

        # Agrupamos por variante para resumir el rendimiento de cada una
        tabla_agrupada = df.groupby('variante')
        metricas = tabla_agrupada.agg(
            cantidad=('feedback', 'count'),
            puntuacion=('feedback', 'mean')
        )
        resumen_final = metricas.reset_index()
        
        print("\n✅ Resumen preliminar del experimento:")
        print(resumen_final)


def update_response():
    """
    Carga y muestra en pantalla el texto de la siguiente respuesta a evaluar.
    """
    # Extraemos el texto de la respuesta actual de la tabla
    texto_actual = df.iloc[response_index]['respuesta']
    
    # Formateamos el texto como párrafo HTML
    if pd.notna(texto_actual):
        formato_html = "<p>" + texto_actual + "</p>"
    else:
        formato_html = "<p>Sin respuesta registrada.</p>"
        
    response.value = formato_html
    
    # Actualizamos la etiqueta con la respuesta actual sobre el total
    total_respuestas = len(df)
    nro_actual = response_index + 1
    count_label.value = f"Respuesta: {nro_actual}/{total_respuestas}"

### Interfaz de Usuario

Ejecuta la siguiente celda para iniciar la evaluación interactiva:

In [ ]:
# Creamos los elementos visuales (widgets) de la interfaz de usuario
response = widgets.HTML()
count_label = widgets.Label()

# Inicializamos mostrando la primera respuesta del lote
update_response()

# Creamos los botones de thumbs up/down y les asignamos la función de evento
boton_positivo = widgets.Button(description='👍')
boton_positivo.on_click(on_button_clicked)

boton_negativo = widgets.Button(description='👎')
boton_negativo.on_click(on_button_clicked)

# Agrupamos los botones de forma horizontal en un contenedor
caja_botones = widgets.HBox([boton_negativo, boton_positivo])

# Presentamos la interfaz en el cuaderno
print("👇 Evaluá cada una de las respuestas de forma ciega usando los botones:")
display(response, caja_botones, count_label)

## 6. Análisis de Resultados

### Métricas de Rendimiento

Una vez completada la evaluación, analizamos los resultados en detalle.

In [ ]:
# Cargamos el archivo final con las respuestas calificadas por el evaluador
results_df = pd.read_csv("resultados.csv")

# Agrupamos por variante
agrupado_variante = results_df.groupby('variante')

# Calculamos estadísticas clave: cantidad (count), éxitos (sum), promedio (mean) y desviación estándar (std)
# Separamos el redondeo para evitar method chaining
metricas_detalladas = agrupado_variante.agg({
    'feedback': ['count', 'sum', 'mean', 'std']
})
estadisticas_redondeadas = metricas_detalladas.round(3)

print("📊 Estadísticas detalladas por variante:")
print(estadisticas_redondeadas)

print("\n✅ Tasa de aprobación:")
for variante in ['A', 'B']:
    datos_variante = results_df[results_df['variante'] == variante]
    tasa_aprobacion = datos_variante['feedback'].mean() * 100
    print(f"  • Variante {variante}: {tasa_aprobacion:.1f}%")

# ── Cálculo de Significancia Estadística (Evitando NameError de p-valor) ──
# Importamos la librería estadística para evaluar si la diferencia es significativa
from scipy import stats

# Filtramos las calificaciones individuales de cada grupo
calificaciones_A = results_df[results_df['variante'] == 'A']['feedback']
calificaciones_B = results_df[results_df['variante'] == 'B']['feedback']

# Realizamos un test t de Student para dos muestras independientes (con varianzas desiguales)
t_resultado, p_value = stats.ttest_ind(calificaciones_A, calificaciones_B, equal_var=False)

print(f"\nSignificancia estadística del experimento:")
print(f"  • p-valor obtenido: {p_value:.4f}")

if p_value < 0.05:
    print("  ✓ La diferencia ES estadísticamente significativa (p < 0.05).")
else:
    print("  ✗ La diferencia NO es estadísticamente significativa (p >= 0.05). Podría deberse al azar.")

### Comparación Visual

In [ ]:
import matplotlib.pyplot as plt

# Configuramos la figura con dos gráficos en paralelo (subplots)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 1. Gráfico de barras para la Tasa de Aprobación
grupo_variante = results_df.groupby('variante')
tasa_aprobacion_serie = grupo_variante['feedback'].mean()
approval_rates = tasa_aprobacion_serie * 100

ax1.bar(approval_rates.index, approval_rates.values, color=['#FF6B6B', '#4ECDC4'])
ax1.set_title('Tasa de Aprobación por Variante', fontsize=14, pad=30)
ax1.set_ylabel('Porcentaje de Aprobación (%)')
ax1.set_ylim(0, 100)

# Agregamos los valores numéricos sobre cada barra
for i, v in enumerate(approval_rates.values):
    ax1.text(i, v + 2, f'{v:.1f}%', ha='center')

# 2. Gráfico de distribución para analizar la frecuencia de calificaciones
for var in ['A', 'B']:
    filtro_variante = results_df[results_df['variante'] == var]
    var_feedback = filtro_variante['feedback']
    ax2.hist(var_feedback, bins=2, alpha=0.7, label=f'Variante {var}')
    
ax2.set_title('Distribución de Calificaciones', fontsize=14, pad=30)
ax2.set_xlabel('Calificación (0 = Negativa, 1 = Positiva)')
ax2.set_ylabel('Frecuencia')
ax2.legend()
ax2.set_xticks([0, 1])

# Ajustamos márgenes para que no se superpongan los títulos
plt.tight_layout()
fig.subplots_adjust(top=0.85)

print("📈 Renderizando gráficos comparativos...")
plt.show()

## 7. Conclusiones y Aprendizajes

### Insights Obtenidos

Basándonos en los resultados del experimento, podemos extraer las siguientes conclusiones:

In [ ]:
# ── Análisis Comparativo Final ──
print("📋 RESUMEN EJECUTIVO DEL EXPERIMENTO A/B")
print("=" * 50)

# Calculamos los promedios de aprobación de cada grupo de forma aislada
filas_A = results_df[results_df['variante'] == 'A']
approval_A = filas_A['feedback'].mean() * 100

filas_B = results_df[results_df['variante'] == 'B']
approval_B = filas_B['feedback'].mean() * 100

print(f"\n📊 Resultados Finales:")
print(f"  • Variante A (Zero-shot): {approval_A:.1f}% de aprobación")
print(f"  • Variante B (Few-shot): {approval_B:.1f}% de aprobación")

# Determinamos el ganador basándonos en la tasa de aprobación
if approval_B > approval_A:
    diferencia = approval_B - approval_A
    print(f"\n🏆 Ganador: Variante B (Few-shot)")
    print(f"   Mejora del {diferencia:.1f}% sobre la variante A")
elif approval_A > approval_B:
    diferencia = approval_A - approval_B
    print(f"\n🏆 Ganador: Variante A (Zero-shot)")
    print(f"   Mejora del {diferencia:.1f}% sobre la variante B")
else:
    print(f"\n🤝 Empate: Ambas variantes tienen el mismo rendimiento")

### Interpretación de Resultados

**¿Por qué una estrategia puede ser mejor que otra?**

1. **Si ganó Few-shot (B)**:
   - Los ejemplos proporcionan contexto valioso
   - El modelo comprende mejor el formato esperado
   - Se reduce la ambigüedad en la tarea

2. **Si ganó Zero-shot (A)**:
   - El modelo es suficientemente capaz sin ejemplos
   - Los ejemplos podrían estar limitando la creatividad
   - La tarea es lo suficientemente clara sin contexto adicional

### Recomendaciones

Basándonos en este experimento:

In [ ]:
# ── Recomendaciones según los resultados y el p-valor ──
print("💡 RECOMENDACIONES:")
print("=" * 30)

if approval_B > approval_A and p_value < 0.05:
    print("\n1. ✅ Usar Few-shot prompting para tareas de generación de nombres.")
    print("2. 📝 Proporcionar 2-3 ejemplos de alta calidad.")
    print("3. 🎯 Asegurar que los ejemplos cubran diferentes estilos.")
elif approval_A > approval_B and p_value < 0.05:
    print("\n1. ✅ Usar Zero-shot prompting para mayor eficiencia.")
    print("2. 📝 Enfocarse en descripciones claras y precisas en las instrucciones.")
    print("3. 🎯 Confiar en las capacidades nativas del modelo.")
else:
    print("\n1. 🤷 Ambas estrategias tienen rendimientos similares en este nivel de muestra.")
    print("2. 📝 Elegir basándose en costo y latencia (Zero-shot suele ser más rápido y consume menos tokens).")
    print("3. 🎯 Considerar realizar una prueba con una muestra mayor para confirmar tendencias.")

### Próximos Pasos

Para mejorar este experimento, considera:

1. **Aumentar el tamaño de la muestra**: Más iteraciones = resultados más confiables
2. **Probar con diferentes tipos de productos**: Validar si los resultados son consistentes
3. **Experimentar con más variantes**: Probar diferentes números de ejemplos (1, 2, 3+)
4. **Analizar el contenido generado**: Evaluar creatividad, coherencia y relevancia
5. **Medir tiempo de respuesta**: Comparar la eficiencia de cada estrategia

### Lecciones Aprendidas

Este experimento demuestra:
- La importancia del A/B Testing en la optimización de prompts
- Cómo evaluar objetivamente diferentes estrategias de prompting
- El valor de la evaluación humana en tareas creativas
- La necesidad de análisis estadístico para conclusiones válidas

**¡Felicitaciones!** Has completado exitosamente un experimento de A/B Testing aplicado a prompting de modelos de lenguaje. 🎉


> **✎ Mentor reversible (bonus).** Hasta acá fuiste quien aprende. Ahora dalo vuelta: elegí la función `get_response(prompt)` o la lógica de la interfaz interactiva en `on_button_clicked(b)`, y escribí un comentario detallado arriba explicando paso a paso a un compañero que recién empieza qué hace y para qué sirve cada línea. Si esa persona imaginaria lo entiende, es porque vos ya lo entendiste de verdad.


### 🧭 Diario de Navegación

Cerramos mirando hacia adentro, no hacia el código. Respondé en una o dos líneas, para vos:

1. ¿Qué concepto de hoy te costó más encajar en la cabeza? ¿Por qué creés que se resistió?
2. Si tuvieras que explicarle este cuaderno a un colega en lo que dura un viaje en ascensor, ¿qué le dirías?

No hay respuesta correcta. Lo que escribas es el mapa de tu propio recorrido.